In [ ]:
# Imports, Configre Pandas
import pandas as pd
from sqlalchemy import create_engine, text, inspect

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Reduce Data and Count initial Rows
ENGINE_URL = 'postgresql://markfinlay:@localhost:5432/osm'

# NOTES for Methodology regarding EPSG:3035 -  accurate distance calculations using meters and satellite assets

# EPSG:3035, also known as "ETRS89 / LAEA Europe" (Lambert Azimuthal Equal-Area), is an equal-area projection. 
# This is its most significant advantage for many analytical tasks.

# When performing operations like calculating ST_Length(way) (when creating filtered_planet_osm_line and later in OSM Data.py for segmentation) 
# or if doing any area-based analysis, an equal-area projection ensures that these measurements are accurate across your 
# study area (Ireland and UK). Geographic coordinate systems like WGS84 (EPSG:4326), which use degrees, are not 
# suitable for accurate distance or area measurements directly.

# Why it matters:
# Trail Segmentation: OSM Data.py script segments trails into fixed lengths (e.g., segment_length = 320 meters). 
# Performing this segmentation accurately requires the trail geometries to be in a coordinate system where distances are measured in meters.

# Bounding Box for Satellite Imagery: When fetching Sentinel-2 imagery in Sentinel 2 Data.py, bounding boxs around the centroid of each 
# trail segment. Specify half_side = 160 (meters) to create a 320m x 320m bounding box. 
# These dimensions must be in meters, and the centroid coordinates (center.x, center.y) must also be in a meter-based system for 
# this to work correctly. EPSG:3035 provides this.

# When requesting data from Sentinel Hub using a bounding box defined in a specific CRS (like EPSG:3035 for your seg_bbox), 
# and specify an image size in pixels (e.g., 32x32 pixels for a 320m x 320m area at 10m/pixel resolution), 
# the service handles the spatial gridding correctly based on that CRS. 
# Using a meter-based projected CRS like EPSG:3035 for defining this request ensures that pixel resolution (10 meters per pixel) 
# directly corresponds to the ground sampling distance intended.

In [ ]:
try:
    engine = create_engine(ENGINE_URL)
    with engine.connect() as connection:
        print("✅ Successfully connected to the database.")

        # Check for PostGIS extension
        postgis_check = connection.execute(text("SELECT 1 FROM pg_extension WHERE extname = 'postgis';")).scalar_one_or_none()
        if postgis_check:
            print("✅ PostGIS extension is enabled.")
        else:
            print("⚠️ WARNING: PostGIS extension is NOT enabled. This notebook heavily relies on it.")
            print("   Please ensure PostGIS is installed and enabled in the database (setup_db.py should handle this).")

        # Check for HSTORE extension
        hstore_check = connection.execute(text("SELECT 1 FROM pg_extension WHERE extname = 'hstore';")).scalar_one_or_none()
        if hstore_check:
            print("✅ HSTORE extension is enabled.")
        else:
            print("⚠️ WARNING: HSTORE extension is NOT enabled.")
            print("   This might be an issue if osm2pgsql was expected to use it (setup_db.py calls osm2pgsql with --hstore).")

except Exception as e:
    print(f"❌ ERROR: Failed to connect to the database or execute initial checks.")
    print(f"   Details: {e}")
    print("   Please ensure PostgreSQL is running, the database '{DB_NAME}' exists, and connection parameters are correct.")
    print("   You might need to run the `setup_db.py` script first.")
    # Optionally, raise the exception if you want the notebook to stop here on failure
    # raise
else:
    print("\nInitial database checks complete. Proceeding with notebook execution...")
    # Configure Pandas options (from your original first cell)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)

In [ ]:
print("\n--- Checking for Essential OSM Tables and Data --- \n")

essential_tables = {
    "planet_osm_point": "Points imported by osm2pgsql.",
    "planet_osm_line": "Lines (roads, paths, etc.) imported by osm2pgsql. CRITICAL for this notebook.",
    "planet_osm_polygon": "Polygons imported by osm2pgsql.",
    "planet_osm_roads": "Roads (subset) imported by osm2pgsql."
}

all_essential_tables_present = True
planet_osm_line_has_data = False

if 'engine' in locals() and engine is not None: # Check if engine exists
    try:
        inspector = inspect(engine)
        with engine.connect() as connection:
            for table_name, description in essential_tables.items():
                if inspector.has_table(table_name, schema='public'):
                    count_result = connection.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar_one()
                    print(f"✅ Table '{table_name}' exists and has {count_result:,} rows. ({description})")
                    if table_name == "planet_osm_line" and count_result > 0:
                        planet_osm_line_has_data = True
                else:
                    print(f"❌ ERROR: Essential table '{table_name}' does NOT exist. ({description})")
                    all_essential_tables_present = False

            if not all_essential_tables_present:
                print("\nOne or more essential OSM tables are missing. The `setup_db.py` script might not have completed successfully.")
            elif not planet_osm_line_has_data:
                print("\n⚠️ WARNING: Table 'planet_osm_line' exists but is empty. Check the osm2pgsql import in `setup_db.py`.")
            else:
                # Optional: More specific check for relevant data in planet_osm_line
                relevant_lines_query = """
                    SELECT COUNT(*) FROM planet_osm_line
                    WHERE (highway IN ('path', 'footway', 'track') OR route = 'hiking')
                    AND surface IS NOT NULL;
                """
                relevant_lines_count = connection.execute(text(relevant_lines_query)).scalar_one()
                if relevant_lines_count > 0:
                    print(f"✅ Found {relevant_lines_count:,} potentially relevant lines in 'planet_osm_line' (with highway/route and surface tags).")
                else:
                    print("⚠️ WARNING: 'planet_osm_line' has data, but no lines match the initial filtering criteria (highway/route and surface tags).")
                    print("   This could mean the imported PBF files lack these specific tags, or there's an issue with the OSM data itself.")

    except Exception as e:
        print(f"❌ ERROR during table checks: {e}")
        print("   Ensure the database connection is stable and the user has permissions.")
else:
    print("❌ ERROR: Database engine ('engine') not initialized from the previous cell. Cannot perform table checks.")

if not all_essential_tables_present or not planet_osm_line_has_data:
    print("\n🛑 CRITICAL CHECKS FAILED. Review messages above before proceeding with the rest of the notebook.")
else:
    print("\nEssential table checks passed. You should be good to proceed.")
print("-" * 50)

In [ ]:
# Define the new table name
new_filtered_table_name = "filtered_planet_osm_line"

# SQL query to create the new table with the filtered data
create_new_table_query = f"""
DROP TABLE IF EXISTS {new_filtered_table_name}; -- Optional: Drop if it exists, for idempotency

CREATE TABLE {new_filtered_table_name} AS
SELECT
    osm_id,
    name,
    surface,
    highway,
    route,
    width,
    tags,
    way,
    ST_Length(way) AS length_m
FROM
    planet_osm_line
WHERE
    (highway IN ('path', 'footway', 'track') OR route = 'hiking')
    AND way IS NOT NULL
    AND NOT ST_IsEmpty(way)
    AND ST_Length(way) > 320
    AND surface IS NOT NULL;
"""

# Optional: Add an index on the geometry column for faster spatial queries later
create_index_query = f"""
CREATE INDEX IF NOT EXISTS idx_{new_filtered_table_name}_way
ON {new_filtered_table_name} USING GIST (way);

-- Optional: Index on osm_id if you frequently query by it
CREATE INDEX IF NOT EXISTS idx_{new_filtered_table_name}_osm_id
ON {new_filtered_table_name} (osm_id);
"""

# engine = create_engine('postgresql://markfinlay:@localhost:5432/osm') # Already defined

with engine.begin() as conn:
    print(f"Creating new table '{new_filtered_table_name}' with filtered data...")
    conn.execute(text(create_new_table_query))
    print(f"Table '{new_filtered_table_name}' created successfully.")
    print(f"Creating indexes on '{new_filtered_table_name}'...")
    conn.execute(text(create_index_query))
    print("Indexes created.")

print("Filtered data has been copied to the new table.")

# Now, your query to get the DataFrame should use the new table
query_from_new_table = f"""
    SELECT name, surface, width, tags, way, length_m -- length_m is already calculated
    FROM {new_filtered_table_name}
    ORDER BY osm_id;
"""

df = pd.read_sql(query_from_new_table, engine)
print(f"Number of rows in the new filtered table '{new_filtered_table_name}': {len(df)}")
df.count()

In [ ]:
df.head()

In [ ]:
df['length_m'].describe()

In [ ]:
df['surface'].value_counts()

In [ ]:
# Have a close look at Wikclow Hills Data 
# ST_MakeEnvelope(<min_lon>, <min_lat>, <max_lon>, <max_lat>, 4326)

# The use of ST_Transform is important as way column in filtered_planet_osm_line is not in EPSG:3857. 
# Given that osm2pgsql usually imports data in Web Mercator (EPSG:3857) this should work correctly. 

query = """
    SELECT osm_id, name, highway, surface, way, ST_Length(way) AS length_m
    FROM filtered_planet_osm_line
    WHERE ST_Intersects( way, ST_Transform(ST_MakeEnvelope(-6.50, 52.9, -6.2, 53.24, 4326), 3857));
    """

gdf = gpd.read_postgis(query, engine, geom_col='way')
# Convert to WGS84 so that the GeoJSON is in longitude/latitude
gdf = gdf.to_crs(epsg=4326)
gdf.explore(zoom_start=11, location=(53.1, -6.3))

In [ ]:
# 2. Create the 'surface_group' column and populate it
# This is Data cleaning and feature engineering to a degree

from sqlalchemy import create_engine, text

with engine.begin() as conn:
    # a) Add the column if it doesn’t already exist
    print("Adding 'surface_group' column to 'filtered_planet_osm_line' ...")
    conn.execute(text("ALTER TABLE filtered_planet_osm_line ADD COLUMN IF NOT EXISTS surface_group text;"))

    # b) Update the column using a CASE expression
    conn.execute(text("""
        UPDATE filtered_planet_osm_line
        SET surface_group = CASE
            WHEN lower(surface) IN (
                'asphalt', 'asphaltsidewalk', 'asphalt;paved', 
                'tarmac', 'bituminous', 'macadam'
            )
            THEN 'asphalt'
            WHEN lower(surface) IN (
                 'concrete', 'concrete asphalt', 'concrete:plates',
                 'concrete:lanes', 'concrete_slabs', 'concrete_with_bitumen', 'cement',
                'unpaved/paved', 'paved', 'paving_stones', 'paving_stones:30',
                'brick', 'bricks', 'paving_stones', 'chipseal',
                'cobblestone', 'cobblestone:flattened'
            )
            THEN 'paved'

            WHEN lower(surface) IN (
                 'gravel', 'fine_gravel', 'gravel,_dirt', 'gravel_and_rock', 'aggregate',
                 'sand/gravel/stones', 'ground;gravel', 'compacted',  'sand;pebbles'
               )
            THEN 'gravel'

            WHEN lower(surface) IN (
                 'dirt', 'dust', 'earth', 'earth,_grass', 'dirt/sand',
                 'dirt; gravel; dirt', 'unpaved', 
                 'ground', 'clay', 'soil'
               )
            THEN 'mud/dirt'

            WHEN lower(surface) IN (
                 'grass', 'grass/trees', 'grass_paver', 'green'
               )
            THEN 'grass'

            ELSE NULL
        END
        WHERE surface IS NOT NULL;
    """))

print("Done!")

In [ ]:
# Count instances per surface_group
df_surface_groups = pd.read_sql("""
    SELECT surface_group, COUNT(*) AS count
    FROM filtered_planet_osm_line
    WHERE surface IS NOT NULL
    GROUP BY surface_group
    ORDER BY count DESC
""", engine)
df_surface_groups

In [ ]:
# Simple bar chart of these frequencies
query = """
    SELECT osm_id, highway, surface, surface_group, tags, way, ST_Length(way) AS length_m
    FROM filtered_planet_osm_line
    WHERE surface IS NOT NULL
    AND surface_group IS NOT NULL
    """

gdf = gpd.read_postgis(query, engine, geom_col='way')
print(gdf.count())

gdf['surface_group'].value_counts().plot.bar(figsize=(8, 4))

In [ ]:
# Balance the data so that we have an even count matching the smallest class
import pandas as pd

# Define your target count per class
target_count = 15220 # IMPORTANT: Set to 15220, which is the count of the smallest class 'paved' before balancing

# Create a list to collect each balanced group
balanced_frames = []

# Sample randomly by proportionality???

# Iterate over each surface_group
for group, group_data in gdf.groupby('surface_group'):
    if len(group_data) >= target_count:
        # Downsample majority groups
        balanced_group = group_data.sample(n=target_count, random_state=42)
    else:
        # Oversample minority groups (with replacement) to reach target_count
        balanced_group = group_data.sample(n=target_count, replace=True, random_state=42)
    balanced_frames.append(balanced_group)

# Combine all groups back into a single GeoDataFrame
gdf_balanced = pd.concat(balanced_frames, ignore_index=True)

# Verify the result
print(gdf_balanced['surface_group'].value_counts())

In [ ]:
# Display the interactive map of balanced data
gdf_balanced = gdf_balanced.to_crs(epsg=3035)

map_balanced = gdf_balanced.explore(
    tooltip='surface_group',  # Show the surface_group when hovering
    popup=True,               # Enable popup on click (shows all attributes)
    tiles='OpenStreetMap'     # You can change the base map style if desired
)

map_balanced

In [ ]:
gdf_balanced.head(10)

In [ ]:
# Write the balanced GeoDataFrame to a new table in the database.
gdf_balanced = gdf_balanced.to_crs(epsg=3035)

gdf_balanced.to_postgis('balanced_planet_osm_line', engine, if_exists='replace', index=False)
print("New table 'balanced_planet_osm_line' has been created and the balanced data is stored.")

In [ ]:
# Select all rows from the balanced table
query = "SELECT * FROM balanced_planet_osm_line;"

# Load the data into a GeoDataFrame.
# Adjust the 'geom_col' parameter if your geometry column is named differently.
balanced_gdf = gpd.read_postgis(query, engine, geom_col='way')

# Display the first few rows
balanced_gdf.head()

In [ ]:
# Descriptive statistics for numerical columns (e.g., length_m, width if numeric)
print("Descriptive statistics for numerical columns:")
balanced_gdf['length_m'].describe()

In [ ]:
# Count of each class in the 'surface_group' column
print("\nDistribution of 'surface_group':")
balanced_gdf['surface_group'].value_counts()

In [ ]:
# group-wise statistics for a specific column, you can use groupby.
# For example, the mean length_m per surface_group:
print("\nMean 'length_m' per surface_group:")
balanced_gdf.groupby('surface_group')['length_m'].mean()

In [ ]:
# Calculate total 'length_m' per surface_group
print("\nTotal 'length_m' per surface_group in balanced_gdf:")
total_length_per_group = balanced_gdf.groupby('surface_group')['length_m'].sum()
print(total_length_per_group)

# Optional: Display as a bar chart for easier visual comparison
import matplotlib.pyplot as plt
total_length_per_group.sort_values(ascending=False).plot.bar(figsize=(10, 5))
plt.title('Total Length of Original OSM Ways per Surface Group (After Balancing by Count)')
plt.ylabel('Total Length (meters)')
plt.xlabel('Surface Group')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# !!! balanced_planet_osm_line is not actually balanced! Further balancing will be done after segemnts have been created

# Process each trail from the GeoDataFrame
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from shapely.geometry import LineString
import geopandas as gpd
from sentinelhub import CRS, BBox

# -------------------------------
# Parameters
# -------------------------------
segment_length = 320  # length in meters for each segment
fixed_width = 320     # width of bounding box (meters)
fixed_height = 320    # height of bounding box (meters)
half_width = fixed_width / 2
half_height = fixed_height / 2

# -------------------------------
# Read Balanced Trails
# -------------------------------
# Query the balanced table
balanced_query = "SELECT osm_id, surface_group, way, ST_Length(way) AS length_m FROM balanced_planet_osm_line;"
# Load as GeoDataFrame (assuming the geometry column is named "way")
balanced_gdf = gpd.read_postgis(balanced_query, engine, geom_col='way')

# For segmentation in metric units, ensure geometries are in an appropriate meter-based CRS
# EPSG:3035 (ETRS89 / LAEA Europe) is an equal-area projection for Europe, including the UK and Ireland, 
# in a single coordinate system. It’s often used as a compromise projection for multi-country datasets
balanced_gdf = balanced_gdf.to_crs(epsg=3035)

print(f"Read {len(balanced_gdf)} trails from balanced_planet_osm_line.")

# -------------------------------
# Create the trail_segments Table
# -------------------------------
create_table_sql = """
DROP TABLE IF EXISTS trail_segments;
CREATE TABLE trail_segments (
    id SERIAL PRIMARY KEY,
    filename TEXT,
    osm_id BIGINT,
    surface_group TEXT,
    center_geom geometry(Point,3035),
    image_data BYTEA
);
"""
with engine.begin() as conn:
    conn.execute(text(create_table_sql))
print("New table 'trail_segments' has been created.")

# -------------------------------
# Segment Each Trail and Insert Rows
# -------------------------------
# Define an SQL insert statement.
# We use PostGIS functions ST_SetSRID(ST_MakePoint(...)) for the center point
# and ST_GeomFromText() for the bounding box (using WKT).
insert_sql = """
INSERT INTO trail_segments (filename, osm_id, surface_group, center_geom, image_data)
VALUES (
    :filename,
    :osm_id,
    :surface_group,
    ST_SetSRID(ST_MakePoint(:center_lon, :center_lat), 3035),
    :image_data
);
"""

for idx_row, trail_row in balanced_gdf.iterrows():
    osm_id = trail_row['osm_id']
    trail_metric = trail_row['way'] 
    
    # --- Segment the Trail ---
    num_segments = int(trail_metric.length // segment_length)
    segments = []
    for i in range(num_segments):
        start = trail_metric.interpolate(i * segment_length)
        end = trail_metric.interpolate((i + 1) * segment_length)
        segments.append(LineString([start, end]))
    # Add remaining piece if necessary
    if trail_metric.length % segment_length > 0:
        segments.append(LineString([trail_metric.interpolate(num_segments * segment_length),
                                     trail_metric.coords[-1]]))
    
    # Process each segment.
    for idx_seg, seg in enumerate(segments):
        # Compute the center point of the segment.
        center = seg.centroid
        center_lon, center_lat = center.x, center.y
        # Construct a filename for this segment.
        filename = f"{osm_id}_segment_{idx_seg + 1}"
    
        # Prepare parameters for insertion.
        params = {
            "filename": filename,
            "osm_id": int(osm_id),
            "surface_group": trail_row['surface_group'],
            "center_lon": center_lon,
            "center_lat": center_lat,
            "image_data": None  # placeholder (NULL); you can later update this field.
        }
    
        # Insert the segment row into the trail_segments table.
        with engine.begin() as conn:
            conn.execute(text(insert_sql), params)

print("All trail segments have been processed and stored in 'trail_segments'.")

In [ ]:
# NEW CELL: Balance the 'trail_segments' by number of segments per surface_group

import pandas as pd
import geopandas as gpd
from sqlalchemy import text # engine should already be defined

print("--- Balancing trail_segments ---")

# 1. Read all segments from the 'trail_segments' table
print("Reading all segments from 'trail_segments' table...")
query_all_segments = "SELECT id, filename, osm_id, surface_group, center_geom FROM trail_segments;"
all_segments_gdf = gpd.read_postgis(query_all_segments, engine, geom_col='center_geom')
print(f"Read {len(all_segments_gdf)} total segments.")

if all_segments_gdf.empty:
    print("No segments found in 'trail_segments' table. Skipping balancing.")
else:
    # 2. Calculate current counts of 320m segments per surface_group
    print("\nCurrent segment counts per surface_group in 'trail_segments':")
    segment_counts_per_group = all_segments_gdf['surface_group'].value_counts()
    print(segment_counts_per_group)

    if segment_counts_per_group.empty:
        print("No surface_group data to balance. Skipping.")
    else:
        # 3. Determine the target count for balancing (e.g., count of the smallest class)
        target_segment_count = segment_counts_per_group.min()
        print(f"\nTarget segment count for balancing (smallest class size): {target_segment_count}")

        balanced_segment_frames = []
        for group, group_data in all_segments_gdf.groupby('surface_group'):
            if len(group_data) >= target_segment_count:
                # Downsample majority groups
                balanced_group = group_data.sample(n=target_segment_count, random_state=42)
            else:
                # Oversample minority groups (with replacement)
                # This path will only be taken if you set target_segment_count to something other than .min()
                # or if you want to ensure even the smallest class reaches a certain threshold.
                # For now, with .min(), only downsampling will occur.
                balanced_group = group_data.sample(n=target_segment_count, replace=True, random_state=42)
            balanced_segment_frames.append(balanced_group)

        if balanced_segment_frames:
            final_balanced_segments_gdf = pd.concat(balanced_segment_frames, ignore_index=True)
            print("\nCounts per surface_group after final balancing of segments:")
            print(final_balanced_segments_gdf['surface_group'].value_counts())

            # 4. Write the finally balanced segments to a new table
            final_balanced_table_name = "final_balanced_trail_segments"
            print(f"\nWriting {len(final_balanced_segments_gdf)} balanced segments to new table '{final_balanced_table_name}'...")

            # Ensure it's in the correct CRS before writing (should inherit from trail_segments' center_geom)
            # but good to be explicit if trail_segments was created with a specific CRS like 3035 for center_geom.
            # final_balanced_segments_gdf = final_balanced_segments_gdf.to_crs(epsg=3035) # If needed

            with engine.connect() as conn: # Drop if exists for idempotency
                conn.execute(text(f"DROP TABLE IF EXISTS {final_balanced_table_name};"))
                conn.commit() # Commit drop before creating
            
            final_balanced_segments_gdf.to_postgis(
                final_balanced_table_name,
                engine,
                if_exists='append', # Use 'append' because we just dropped it, or 'replace' if you don't manually drop
                index=False,
                # Explicitly define geometry type and srid if needed, though usually inferred
                # dtype={'center_geom': 'geometry(Point,3035)'}
            )
            print(f"Table '{final_balanced_table_name}' created with balanced segments.")

            # Optional: Add an index
            with engine.begin() as conn:
                 conn.execute(text(f"CREATE INDEX IF NOT EXISTS idx_{final_balanced_table_name}_geom ON {final_balanced_table_name} USING GIST (center_geom);"))
                 conn.execute(text(f"CREATE INDEX IF NOT EXISTS idx_{final_balanced_table_name}_sg ON {final_balanced_table_name} (surface_group);"))
            print(f"Indexes created on '{final_balanced_table_name}'.")

        else:
            print("No balanced segment frames created. Final table not written.")

In [ ]:
# Segements are Balanced!
# Select all rows from trail_segments
query = "SELECT * FROM final_balanced_trail_segments;"

# Load the data into a GeoDataFrame.
trail_segments_gpdf= gpd.read_postgis(query, engine, geom_col='center_geom')

# Display the first few rows
trail_segments_gpdf.head()

In [ ]:
from matplotlib.patches import Rectangle

# Process each trail from the GeoDataFrame
sampled_trails = balanced_gdf.sample(n=20, random_state=42)  
for idx_row, trail_row in sampled_trails.iterrows():
    osm_id = trail_row['osm_id']
    trail_metric = trail_row['way']
    
    # --- Visualization ---
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot the original trail in light gray
    x, y = trail_metric.xy
    ax.plot(x, y, color='lightgray', linewidth=2, label='Original Trail')

    # Select all segments by osm_id and load them into a GeoDataFrame
    query = f"SELECT * FROM trail_segments WHERE osm_id = {osm_id}"
    trail_segments_gpdf = gpd.read_postgis(query, engine, geom_col='center_geom').to_crs(epsg=3035)

    # Plot the centroids from the trail segments
    ax.scatter(trail_segments_gpdf['center_geom'].x, 
               trail_segments_gpdf['center_geom'].y, 
               color='red', s=50, label='Trail Segment Centroid')
    
    # For each centroid, add a 320m x 320m bounding box centered on it
    for idx, row in trail_segments_gpdf.iterrows():
        centroid = row['center_geom']
        lower_left_x = centroid.x - 160  # 320/2
        lower_left_y = centroid.y - 160  # 320/2
        rect = Rectangle((lower_left_x, lower_left_y), 320, 320, 
                         linewidth=1, edgecolor='blue', facecolor='none')
        ax.add_patch(rect)
    
    ax.set_aspect('equal', 'box')
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_title(f"Trail with 320m x 320m Bounding Boxes (OSM id: {osm_id})")
    ax.legend()
    plt.show()
